# Guide 5: The transformer: blocks, stacks and a model that works

*How LLMs work* · [Read the guide](https://www.vinaygautam.com/blog/how-llms-work/05-transformer) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/05-transformer.ipynb)

This notebook holds every code block from Guide 5, in the guide's order: 10 code cells, 9 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- The pieces so far, and what's missing — no code
- A network for each token — no code
- Residual connections — no code
- Normalization — 2 code cells
- The block, assembled — 1 code cell
- Stacking: the whole model — 2 code cells
- A transformer that works — 4 code cells
- Taking pieces away — 1 code cell
- What carries forward — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## Normalization

The code in this guide uses PyTorch 2.14.1, pinned in the series `requirements.txt` as of October 2026. Earlier guides used numpy and wrote backpropagation out by hand. PyTorch supplies ready-made layers and **autograd**, which records each operation in the forward pass and then runs backpropagation for you. Before trusting it with a transformer, check it against a result you know. Recall Guide 1's two-weight network, with $x = 1$, $y = 2$, $w_1 = 2$ and $w_2 = 0.5$. By hand, the loss was 0.5 and the gradients were −0.5 for $w_1$ and −2 for $w_2$.

In [2]:
import torch
w1 = torch.tensor(2.0, requires_grad=True)     # ask autograd to track these two
w2 = torch.tensor(0.5, requires_grad=True)
x, y = 1.0, 2.0
y_hat = w2 * torch.relu(w1 * x)
loss = 0.5 * (y_hat - y) ** 2                  # squared error, as in Guide 1
loss.backward()                                # backpropagation
print(loss.item(), w1.grad.item(), w2.grad.item())

0.5 -0.5 -2.0


The numbers match. From here on, `loss.backward()` does the chain rule. Now the layer normalization, with $\epsilon$ set to 0 so the output matches the hand calculation:

In [3]:
row = torch.tensor([2.0, 4.0, 6.0, 8.0])
print(torch.nn.functional.layer_norm(row, (4,), eps=0))    # γ = 1 and β = 0

tensor([-1.3416, -0.4472,  0.4472,  1.3416])


## The block, assembled

The `attention` method is Guide 4's computation for every head at once. One layer, `qkv`, produces all the heads' queries, keys and values side by side in a row of $3\,d_{\text{model}}$ numbers. `view` and `permute` only cut that row into three slices of $h$ heads each and move the head axis forward. `triu(1)` keeps the $-\infty$ entries above the diagonal, which is the causal mask. The last line puts the heads' outputs side by side again and applies $W^O$, Guide 4's matrix that recombines them.

In [4]:
import math
import torch.nn as nn
import torch.nn.functional as F
class Block(nn.Module):
    def __init__(self, d_model, h, d_ff, residual=True, norm=True):
        super().__init__()
        self.h, self.residual = h, residual
        self.ln1 = nn.LayerNorm(d_model) if norm else nn.Identity()
        self.ln2 = nn.LayerNorm(d_model) if norm else nn.Identity()
        self.qkv = nn.Linear(d_model, 3 * d_model)     # W^Q, W^K, W^V of every head, side by side
        self.proj = nn.Linear(d_model, d_model)        # W^O
        self.fc1 = nn.Linear(d_model, d_ff)            # W_1, b_1
        self.fc2 = nn.Linear(d_ff, d_model)            # W_2, b_2
    def attention(self, x):                            # x: B × n × d_model
        B, n, d_model = x.shape
        d_k = d_model // self.h
        Q, K, V = self.qkv(x).view(B, n, 3, self.h, d_k).permute(2, 0, 3, 1, 4)   # each B × h × n × d_k
        S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)                              # scores, B × h × n × n
        M = torch.full((n, n), float("-inf")).triu(1)                             # causal mask, n × n
        self.A = torch.softmax(S + M, dim=-1)                                     # weights, kept to look at later
        return self.proj((self.A @ V).transpose(1, 2).reshape(B, n, d_model))     # heads side by side, then W^O
    def mlp(self, x):                                  # x: B × n × d_model
        return self.fc2(F.gelu(self.fc1(x)))           # widths d_model → d_ff → d_model
    def forward(self, x):                              # B × n × d_model in, the same shape out
        a = self.attention(self.ln1(x))
        x = x + a if self.residual else a              # first addition to the stream
        m = self.mlp(self.ln2(x))
        return x + m if self.residual else m           # second addition

## Stacking: the whole model

In code the whole model is a few lines around `Block`. This one is sized for Guide 1's toy text: 27 characters as its vocabulary (single characters, where an LLM's tokens are Guide 2's subword pieces), $d_{\text{model}} = 32$, $h = 4$ heads, $L = 2$ blocks, $d_{\text{ff}} = 128$, and room for 16 characters of context.

In [5]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size=27, d_model=32, h=4, L=2, d_ff=128, n_max=16, **switches):
        super().__init__()
        self.W_E = nn.Embedding(vocab_size, d_model)                 # token embedding table
        self.W_P = nn.Embedding(n_max, d_model)                      # position table
        self.blocks = nn.ModuleList([Block(d_model, h, d_ff, **switches) for _ in range(L)])
        self.ln_f = nn.LayerNorm(d_model) if switches.get("norm", True) else nn.Identity()
        self.out = nn.Linear(d_model, vocab_size)                    # output layer
    def forward(self, ids):                                          # ids: B × n token IDs
        x = self.W_E(ids) + self.W_P(torch.arange(ids.shape[1]))     # B × n × d_model
        for block in self.blocks:
            x = block(x)
        self.stream = x                                              # top of the stream, kept to look at later
        return self.out(self.ln_f(x))                                # logits, B × n × vocab_size
torch.manual_seed(0)
model = TinyGPT()
print(sum(param.numel() for param in model.parameters()))

27739


GPT-2 small is the same structure with larger numbers, and its budget takes a few lines of arithmetic:

In [6]:
def gpt2_small_budget(vocab_size=50257, n_max=1024, d_model=768, d_ff=3072, L=12):
    attention = 4 * d_model**2 + 4 * d_model           # W^Q, W^K, W^V, W^O and their bias rows
    mlp = 2 * d_model * d_ff + d_ff + d_model          # W_1, W_2, b_1, b_2
    norms = 2 * 2 * d_model                            # two layer norms, each a γ row and a β row
    block = attention + mlp + norms
    tables, final_norm = vocab_size * d_model + n_max * d_model, 2 * d_model
    return {"token embeddings, 50,257 × 768": vocab_size * d_model, "position vectors, 1,024 × 768": n_max * d_model,
            "one block: attention": attention, "one block: MLP": mlp, "one block: two layer norms": norms,
            "one block, total": block, "12 blocks": L * block, "final layer norm": final_norm,
            "total": tables + L * block + final_norm}
for name, count in gpt2_small_budget().items():
    print(f"{name:32}{count:>12,}")

token embeddings, 50,257 × 768    38,597,376
position vectors, 1,024 × 768        786,432
one block: attention               2,362,368
one block: MLP                     4,722,432
one block: two layer norms             3,072
one block, total                   7,087,872
12 blocks                         85,054,464
final layer norm                       1,536
total                            124,439,808


## A transformer that works

The code simplifies twice. The optimizer is **Adam** (adaptive moment estimation), a refinement of gradient descent that adapts the step size for each parameter ([Kingma & Ba, 2015](https://arxiv.org/abs/1412.6980)). Treat it as gradient descent for now. Guide 7 explains it. And every step uses all 112 windows, where real training uses a small sample of the data at each step, which Guide 7 covers.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [7]:
text = ("the cat sat on the mat and the dog sat on the log and the "
        "cat saw the dog then the dog ran to the cat and the cat ran on the mat")
vocab = sorted(set("abcdefghijklmnopqrstuvwxyz "))                             # the space is ID 0, as in Guide 1
ids = torch.tensor([vocab.index(c) for c in text])                             # 128 character IDs
n = 16
inputs = torch.stack([ids[start:start + n] for start in range(len(ids) - n)])      # 112 × 16
targets = torch.stack([ids[start + 1:start + n + 1] for start in range(len(ids) - n)])   # the same windows, one step later
def train(model, steps, report=()):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    for step in range(steps + 1):
        logits = model(inputs)                                                 # 112 × 16 × 27
        loss = F.cross_entropy(logits.reshape(-1, len(vocab)), targets.reshape(-1))    # softmax, then mean cross-entropy
        if step in report:
            print(step, f"{loss.item():.3f}")
        if step < steps:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()
final_loss = train(model, 500, report=(0, 50, 100, 200, 500))

0 3.306


50 0.615


100 0.212


200 0.157


500 0.152


An average hides where that gain comes from. The next block splits the loss by position in the window. Position 1 has one character of context, position 16 has sixteen. Beside each loss it prints the floor, the lowest loss any predictor can reach with that much context. It's Guide 4's counting argument, redone on these same 1,792 examples (112 windows of 16 positions): for each context of $k$ characters, count how often each next character follows it. The best possible predictor outputs exactly those frequencies, and the floor is its average cross-entropy.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [8]:
from collections import Counter
def floor(k):                                          # lowest possible loss with k characters of context
    seen = Counter((text[start:start + k], text[start + k]) for start in range(len(inputs)))
    context = Counter()
    for (c, ch), count in seen.items():
        context[c] += count
    return sum(count * math.log(context[c] / count) for (c, ch), count in seen.items()) / len(inputs)
with torch.no_grad():                                  # inference only: no gradients needed
    losses = F.cross_entropy(model(inputs).reshape(-1, len(vocab)), targets.reshape(-1), reduction="none")   # 1,792 losses
by_position = losses.view(len(inputs), n).mean(0)      # 16 values: the 112 × 16 losses averaged over the 112 windows
for k in range(1, 17):
    print(f"{k:2}  model {by_position[k - 1].item():.3f}  floor {floor(k):.3f}")

 1  model 0.874  floor 0.869
 2  model 0.304  floor 0.301
 3  model 0.300  floor 0.298
 4  model 0.206  floor 0.205
 5  model 0.170  floor 0.169
 6  model 0.127  floor 0.125
 7  model 0.110  floor 0.108
 8  model 0.110  floor 0.108
 9  model 0.103  floor 0.101
10  model 0.051  floor 0.050
11  model 0.038  floor 0.037
12  model 0.026  floor 0.025
13  model 0.014  floor 0.012
14  model 0.001  floor 0.000
15  model 0.001  floor 0.000
16  model 0.001  floor 0.000


At every position the model sits almost exactly on the floor, and from 14 characters on both are near zero. A floor of zero means every context of that length in this text has only one continuation. The model has memorized a 128-character text. Ask it to continue "the c", taking the highest-scoring character each time. That's the simplest rule for choosing, and Guide 6 covers the ones LLMs usually run.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [9]:
out = [vocab.index(c) for c in "the c"]
with torch.no_grad():
    while len(out) < 45:
        logits = model(torch.tensor([out[-16:]]))      # the last 16 characters at most, as a batch of one
        out.append(logits[0, -1].argmax().item())      # highest-scoring next character
print("".join(vocab[i] for i in out))

the cat sat on the mat and the dog sat on the


Guide 4's attention weights came from random matrices and meant nothing. These are trained. For the window "the cat sat on t", the code prints each head's average weight on the character just before the current one. For comparison, take a head that spreads its weight evenly. Position $k$ can see $k$ characters, so it would put $1/k$ on the previous one, and the average of $1/2, 1/3, \ldots, 1/16$ is about 0.16.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [10]:
window = torch.tensor([[vocab.index(c) for c in "the cat sat on t"]])
with torch.no_grad():
    model(window)
for number, block in enumerate(model.blocks, start=1):
    A = block.A[0]                                     # h × n × n
    print("block", number, [round(A[head].diagonal(-1).mean().item(), 2) for head in range(4)])

block 1 [0.33, 0.24, 0.46, 0.18]
block 2 [0.19, 0.2, 0.2, 0.42]


## Taking pieces away

Do the residual connections and the normalization earn their place? The `switches` let you test it. The next block trains three 8-block models for 300 steps each: the full block, one with the additions removed, and one with the normalization removed. It ran for about half a minute on a 4-core laptop CPU in October 2026, and yours will differ. It first prints the floor for a predictor with no context, counted on these targets. Then for each model it prints two numbers. The first is measured before training: the standard deviation of each slot across the 16 positions, averaged, then divided by the standard deviation of all the numbers at the top of the stream. Near 1 means most of the variation is between positions, and near 0 means every position holds almost the same vector. The second is the loss after training.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [11]:
counts = Counter(targets.reshape(-1).tolist())             # how often each character is a target
print("no-context floor", f"{sum(count * math.log(targets.numel() / count) for count in counts.values()) / targets.numel():.3f}")
for name, switches in [("full block", {}), ("no residual", {"residual": False}), ("no normalization", {"norm": False})]:
    torch.manual_seed(0)
    deep = TinyGPT(L=8, **switches)
    with torch.no_grad():
        deep(inputs)                                   # fills deep.stream, B × n × d_model
    spread = (deep.stream.std(dim=1).mean() / deep.stream.std()).item()    # std over the n positions (B × d_model), relative to overall
    print(f"{name:18}{spread:8.3f}{train(deep, 300):8.3f}")

no-context floor 2.318


full block           0.910   0.153


no residual          0.030   2.319


no normalization     0.809   0.152


---

← [Guide 4: Attention](04-attention.ipynb) · [Guide 6: Next-token prediction](06-next-token.ipynb) →

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).